In [ ]:
import os
import sys
import json
import numpy as np

# local modules
thisdir = os.getcwd()
topdir = os.path.abspath(os.path.join(thisdir, '../../'))
sys.path.append(topdir)

import tools.iotools as iotools
import tools.dftools as dftools
import tools.omstools as omstools

In [ ]:
# set path to data files

# settings
datadir = '/eos/user/l/llambrec/dialstools-output'
year = '2024'
eras = [
    #'B-v1',
    #'C-v1',
    'D-v1',
    #'E-v1',
    #'E-v2',
    #'F-v1',
    #'G-v1',
    #'H-v1',
    #'I-v1',
    #'I-v2'
]
dataset = 'ZeroBias'
reco = 'PromptReco'
menames = {
    #'EB': 'EcalBarrel-EBOccupancyTask-EBOT digi occupancy',
    #'EE+': 'EcalEndcap-EEOccupancyTask-EEOT digi occupancy EE +',
    #'EE-': 'EcalEndcap-EEOccupancyTask-EEOT digi occupancy EE -',
    'BPix1': 'PixelPhase1-Phase1_MechanicalView-PXBarrel-clusters_per_SignedModuleCoord_per_SignedLadderCoord_PXLayer_1'
}

# find files corresponding to settings
files = {}
for era in eras:
    files[era] = {}
    for melabel, mename in menames.items():
        mainera, version = era.split('-', 1)
        f = f'{dataset}-Run{year}{mainera}-{reco}-{version}-DQMIO-{mename}.parquet'
        f = os.path.join(datadir, f)
        files[era][melabel] = f

# existence check
missing = []
for era, temp in files.items():
    for melabel, f in temp.items():
        if not os.path.exists(f):
            missing.append(f)
if len(missing) > 0:
    raise Exception(f'The following files do not exist: {missing}')

In [ ]:
# set path to OMS files

omsfiles = {}
for era in eras:
    basedir = 'omsdata' # use new files (with ECAL DCS bits)
    #basedir = '../pixel_clusters_2024/omsdata' # use older files (for pixel study)
    omsfile = os.path.join(basedir, f'omsdata_Run{year}{era}.json')
    if not os.path.exists(omsfile):
        print(f'WARNING: OMS file {omsfile} does not exist.')
    omsfiles[era] = omsfile

In [ ]:
# set path to HLT rate files
# (just use the ones from pixel study as we're dealing with the same ZeroBias triggers)

hltfiles = {}
for era in eras:
    #basedir = 'omsdata_test' # use new files (but should be the same as older files?)
    basedir = '../pixel_clusters_2024/omsdata' # use older files (for pixel study)
    hltfile = os.path.join(basedir, f'hltrate_Run{year}{era}.json')
    if not os.path.exists(hltfile):
        print(f'WARNING: HLT rate file {hltfile} does not exist.')
    hltfiles[era] = hltfile

In [ ]:
# check if all lumisections in the data are also in the OMS data

# results (2025/08/22):
# - run 380147, LS 186 (era C) is missing in both old (pixel) and new (ecal) OMS files.
# - no other missing lumisections.

# loop over eras
for era in eras:
    
    # load OMS data lumisections
    with open(omsfiles[era], 'r') as f:
        omsinfo = json.load(f)
    omsdata_run_nbs = np.array(omsinfo['run_number']).astype(int)
    omsdata_ls_nbs = np.array(omsinfo['lumisection_number']).astype(int)
    omsdata_ids = omsdata_run_nbs * 10000 + omsdata_ls_nbs
    
    # loop over monitoring elements
    for melabel, mename in menames.items():
        
        # load data lumisections
        df = iotools.read_parquet(files[era][melabel], columns=['run_number', 'ls_number'])
        data_run_nbs = df['run_number'].values.astype(int)
        data_ls_nbs = df['ls_number'].values.astype(int)
        data_ids = data_run_nbs * 10000 + data_ls_nbs
        
        # find missing lumisections
        oms_missing_mask = np.isin(data_ids, omsdata_ids, invert=True)
        oms_missing = data_ids[oms_missing_mask]
        
        # print results
        print(f'Era {era}, ME {melabel}:')
        print(f'  Missing in OMS: {len(oms_missing)}')
        if len(oms_missing)>0: print(f'  {oms_missing[:10]}')

In [ ]:
# check if all lumisections in the data are also in the HLT rate data

# results (2025/08/22):
# - many missing runs, but it seems to be consistent between the old files and the new.
# - probably happened for pixel study as well, but missed because of running in jobs.
# - seems to be fine though, OMS shows that there is actually no ZeroBias trigger for those runs, so can be ignored.

trigger = 'HLT_ZeroBias_v*'

# loop over eras
for era in eras:
    
    # load HLT rate data
    with open(hltfiles[era], 'r') as f:
        hltinfo = json.load(f)
    
    # loop over monitoring elements
    for melabel, mename in menames.items():
        
        # load data lumisections
        df = iotools.read_parquet(files[era][melabel], columns=['run_number', 'ls_number'])
        data_run_nbs = df['run_number'].values.astype(int)
        data_ls_nbs = df['ls_number'].values.astype(int)
        
        # find trigger rate
        print('---')
        print(f'Now testing {melabel} {era}')
        print('---')
        test = omstools.find_hlt_rate_for_lumisections(data_run_nbs, data_ls_nbs, hltinfo, trigger, verbose=True)